# 06 Trade Log and Slippage

本课把均线策略从净值曲线拆成订单成交和完整交易日志。重点不是增加复杂度，而是看清楚策略的钱到底由哪些交易赚出来。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from quant_trading.execution import (
    build_order_fills,
    build_round_trip_trade_log,
    plot_trade_log,
    summarize_trade_log,
)
from quant_trading.market_data import download_ohlcv
from quant_trading.moving_average import add_moving_average_strategy_next_open

## 1. 下载数据并生成 next-open 仓位

这里先不在均线函数里扣成本，因为本课要把滑点和佣金放到执行层统一处理。

In [ ]:
symbol = "SPY"
short_window = 10
long_window = 200

df = download_ohlcv(symbol=symbol, start="2000-01-01", auto_adjust=True)
strategy = add_moving_average_strategy_next_open(
    df,
    short_window=short_window,
    long_window=long_window,
    transaction_cost_bps=0.0,
)

strategy[["Open", "Close", "ma_short", "ma_long", "signal", "position", "trade"]].tail()

## 2. 生成订单成交明细

`fills` 是单边成交明细。一次买入算一条，一次卖出也算一条。

In [ ]:
slippage_bps = 2.0
commission_bps = 1.0

fills = build_order_fills(
    strategy,
    slippage_bps=slippage_bps,
    commission_bps=commission_bps,
)

fills.head(10)

## 3. 配成完整交易日志

一笔完整交易是一次买入加后续一次卖出。最后一笔如果还没卖出，会按最新开盘价临时估值。

In [ ]:
trade_log = build_round_trip_trade_log(
    strategy,
    slippage_bps=slippage_bps,
    commission_bps=commission_bps,
)

trade_log.head(10)

## 4. 汇总交易表现

这一张表比单纯净值曲线更适合复盘，因为它能显示胜率、最好最差交易、平均持有周期和成本拖累。

In [ ]:
summary = summarize_trade_log(trade_log)
summary

## 5. 看最差交易

策略复盘时，最差交易通常比最好交易更值得研究。它会告诉你策略在什么环境中容易失效。

In [ ]:
trade_log.sort_values("net_return").head(5)[[
    "trade_number",
    "status",
    "entry_date",
    "exit_date",
    "holding_days",
    "gross_return",
    "net_return",
    "cost_drag",
]]

## 6. 画交易收益和累计交易净值

上图看每笔交易赚亏，下图看交易结束后的累计净值。

In [ ]:
plot_trade_log(trade_log);